In [2]:
import pandas as pd

columnas = ["Name", "Sex", "Event", "Equipment", "Age", "BodyweightKg",
            "Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg",
            "TotalKg", "Dots", "Federation", "Date", "MeetCountry"]

df = pd.read_csv("../data/raw/openpowerlifting.csv",
                 usecols=columnas, low_memory=False)

df.shape

(4059940, 14)

In [3]:
df.head()          # primeras 5 filas
df.info()          # tipos de datos y cantidad de nulos
df.describe()      # estadísticas básicas
df["Sex"].value_counts()
df["MeetCountry"].value_counts().head(10)

<class 'pandas.DataFrame'>
RangeIndex: 4059940 entries, 0 to 4059939
Data columns (total 14 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Name             str    
 1   Sex              str    
 2   Event            str    
 3   Equipment        str    
 4   Age              float64
 5   BodyweightKg     float64
 6   Best3SquatKg     float64
 7   Best3BenchKg     float64
 8   Best3DeadliftKg  float64
 9   TotalKg          float64
 10  Dots             float64
 11  Federation       str    
 12  Date             str    
 13  MeetCountry      str    
dtypes: float64(7), str(7)
memory usage: 433.6 MB


MeetCountry
USA          2174542
Russia        432052
Ukraine       112158
Canada         99547
Australia      94574
Germany        90524
England        85036
France         61170
Norway         60381
Finland        56140
Name: count, dtype: int64

In [4]:
# 1. Porcentaje de nulos por columna
(df.isna().mean() * 100).round(1)

Name                0.0
Sex                 0.0
Event               0.0
Equipment           0.0
Age                36.1
BodyweightKg        1.1
Best3SquatKg       30.3
Best3BenchKg       11.3
Best3DeadliftKg    25.1
TotalKg             6.7
Dots                7.3
Federation          0.0
Date                0.0
MeetCountry         0.0
dtype: float64

In [5]:
# 2. Cuántos nombres distintos hay (aproximación de levantadores únicos)
df["Name"].nunique()

1018414

In [6]:
# 3. Estadísticas de edad, peso corporal y total, para detectar valores raros
with pd.option_context("display.float_format", "{:,.1f}".format):
    display(df[["Age", "BodyweightKg", "TotalKg"]].describe())

,Age,BodyweightKg,TotalKg
count,"2,594,094.0","4,015,475.0","3,788,676.0"
mean,30.7,83.8,380.0
std,13.3,22.7,200.9
min,3.5,15.0,1.0
25%,20.5,66.9,210.9
50%,27.0,81.6,360.0
75%,38.0,98.3,532.5
max,105.5,300.0,"1,407.5"


In [7]:
# 4. Cuántos registros tiene Argentina
df[df["MeetCountry"] == "Argentina"].shape

(21788, 14)

# 1 : No se que significa el campo de "Dots"
# 2 : 1M de nombres distintos y 4M de datos, bastantes veces compitieron los mismos
# 3 : Hay algo mal porque no entiendo nada estos resultados
# 4 : Muy pocos resultados, pero me lo esperaba ya que Argentina no es un país que este muy interesado en el powerlifting

In [13]:
# Tipos de eventos: 
# S = Squat
# B = Benchpress
# D = Deadlift 
df["Event"].value_counts()

Event
SBD    2932314
B       804194
D       220458
BD       76241
S        20023
SB        4187
SD        2523
Name: count, dtype: int64

In [14]:
# Porcentaje de nulos en sentadilla según el tipo de competencia.
# Sirve para comprobar si los nulos vienen de competencias que no incluyen sentadilla.
df.groupby("Event")["Best3SquatKg"].apply(lambda s: s.isna().mean() * 100).round(1)

Event
B      100.0
BD     100.0
D      100.0
S        6.4
SB       9.0
SBD      4.4
SD       2.8
Name: Best3SquatKg, dtype: float64

In [15]:
df[df["TotalKg"] < 50] \
    .sort_values("TotalKg") \
    [["Name", "Event", "Age", "BodyweightKg", "TotalKg", "Federation"]] \
    .head(50)

,Name,Event,Age,BodyweightKg,TotalKg,Federation
2694375,Vadim Tokarev,B,54.5,83.00,1.00,FPR
3140216,Veiko Friedrich,B,NaN,80.50,1.00,BVDK
3140196,Sören Radtke,B,21.0,89.30,1.00,BVDK
3140233,Rainer Geese,B,NaN,65.70,1.00,BVDK
3140192,Alex Wambolt,B,22.0,78.50,2.00,BVDK
3140237,Jürgen Groß,B,63.5,78.80,2.00,BVDK
3140183,Chris Konzagk,B,NaN,89.70,2.00,BVDK
2297077,T. McGuire,B,NaN,110.00,2.50,WDFPF
1005861,Raquel Kavilhuka,D,43.5,68.00,2.50,CBLB
1005908,Valdinei Barbosa,B,34.5,81.00,2.50,CBLB


In [16]:
df[df["Event"] == "SBD"] \
    .nsmallest(30, "TotalKg") \
    [["Name", "Event", "Age","TotalKg", "BodyweightKg", "Federation"]]

,Name,Event,Age,TotalKg,BodyweightKg,Federation
3223010,Elliott Peterson,SBD,9.0,7.00,41.60,RPS
3240459,Aiden Giarolo,SBD,8.0,7.00,35.70,RPS
3240474,Aiden Giarolo,SBD,8.0,7.00,35.70,RPS
3240361,Aubree Miller,SBD,9.0,8.00,40.00,RPS
346448,Nikol Malykh,SBD,6.0,24.00,27.20,WRPF
2498548,J. Waltz,SBD,NaN,25.00,48.00,USA-UA
630134,Charles Mosher,SBD,5.0,28.60,22.50,RAW-CAN
3196527,Aria Green,SBD,6.0,34.00,44.00,RPS
2285385,Marcus Murphy,SBD,NaN,34.02,56.00,USPF
3216544,Kinley Perry,SBD,NaN,34.02,19.00,RPS


In [17]:
sbd = df[df["Event"] == "SBD"].dropna(subset=["TotalKg", "BodyweightKg", "Dots"])
print(sbd.duplicated().sum())   # debería dar 108073

dups = sbd[sbd.duplicated(keep=False)].sort_values(["Name", "Date"])
dups[["Name", "Date", "Federation", "BodyweightKg", "TotalKg"]].head(20)

108073


,Name,Date,Federation,BodyweightKg,TotalKg
864440,A'ron Washington,2024-10-05,USAPL,123.90,740.00
864482,A'ron Washington,2024-10-05,USAPL,123.90,740.00
3327043,A. Albanese,1997-09-06,APA,78.93,594.21
3327057,A. Albanese,1997-09-06,APA,78.93,594.21
2447456,A. Alexander,1989-09-30,USPF,67.50,470.00
2447562,A. Alexander,1989-09-30,USPF,67.50,470.00
2261546,A. Atkins,1982-11-20,USPF,67.50,362.87
2261655,A. Atkins,1982-11-20,USPF,67.50,362.87
2492009,A. Baldwin,1980-12-06,USPF,67.50,410.00
2492068,A. Baldwin,1980-12-06,USPF,67.50,410.00


pico de edad: entre hombres y mujeres esta claro que se encuentra entre los 20-30 años
también es de esperar que, por un tema biológico los dots en hombres es mayor que en mujeres
para el readme: lo del tema biologico

mejoria por edad: los competidores mas jovenes son los que van a tener mejor desempeño a medida que mas compiten.
lo que no me esperaba era que (a mi parecer) es bastante la mejoria +60 dots los menores de 20, y entre 20 y 30, +44, yo esperaba que los mejores fuesen los de 20 a 30 años 
para el readme: los que debutan entre 30 y 39 llegan a la quinta competencia con un nivel (374,2) casi igual al de los menores de 20 (373,7).



predicción
argentina contra el mundo: yo opino que argentina tiene un mayor dot por tener menos registros, al tener el resto del mundo muchísimos más (2.5m contra 13k) el promedio tiene que bajar por que los dots mas bajos bajan el promedio.
además, diría que también entran los más interesados y no tanto amateur, pero no estoy tan seguro sobre esto, tengo mis dudas.
y si refinamos mas el análisis (equipamiento, edad, tiempo entre competencias) es también otro el resultado.

predicción
equipamiento: 
aclaración: no se exactamente que tipos de equipamientos entran en cada categoría así que no sabría decir cuanto puede afectar cada uno, pero asumiendo que son solo muñequeras, rodilleras, coderas y ese tipo de cosas que apretan las articulaciones no creo que afecte tanto.
PD: acabo de recordar lo de los trajes y no tengo ni idea que son.

entre 10-15 puntos de dots creo que agrega el equipamiento, siento que es mucho pero esperable.

yo digo que si es lo que pienso (muñequeras etc) donde más ayuda debe ser en sentadillas, y donde menos en banco plano.
deadlift no porque creo que ni usan (no creo que la faja cuente como equipamiento) y simplemente se ponen chalk en las manos

y yo digo que habra mas diferencia entre las mismas personas usando y no usando equipamiento
